In [15]:
import pandas as pd


In [16]:
df = pd.read_csv("gk_qna_dataset.csv")
df

,question,answer
0,"What is the capital of ""France""?",Paris
1,"What is the capital of ""Germany""?",Berlin
2,"What is the capital of ""Italy""?",Rome
3,"What is the capital of ""Spain""?",Madrid
4,"What is the capital of ""Japan""?",Tokyo
...,...,...
166,"Can you tell me, ""Which bird cannot fly""?",Ostrich
167,"I was wondering, ""What is 'H2O' commonly called""?",Water
168,"I was wondering, ""Which planet is known as the...",Mars
169,"Can you tell me, ""What is the main religion in...",Islam


In [17]:
df['question']

,question
0,"What is the capital of ""France""?"
1,"What is the capital of ""Germany""?"
2,"What is the capital of ""Italy""?"
3,"What is the capital of ""Spain""?"
4,"What is the capital of ""Japan""?"
...,...
166,"Can you tell me, ""Which bird cannot fly""?"
167,"I was wondering, ""What is 'H2O' commonly called""?"
168,"I was wondering, ""Which planet is known as the..."
169,"Can you tell me, ""What is the main religion in..."


In [18]:
df['question'][0]

'What is the capital of "France"?'

# Tokenize

In [19]:
import re
def tokenize(text):
  # lowercase
  text = text.lower()
  #remove quotes
  text = re.sub( r"[\"']","",text )
  #remove punctuations
  text = re.sub(r"^a-z0-9\s"," ",text)
  #tokenize plit into words
  tokens = text.split()

  return tokens

In [20]:
print(tokenize(df['question'][0]))

['what', 'is', 'the', 'capital', 'of', 'france?']


In [21]:
vocab = {'<UNK>':0}

In [33]:
def build_vocab(row):
  tokenized_question = tokenize(row['question'])
  tokenized_answer = tokenize(row['answer'])

  marged_tokens = tokenized_question + tokenized_answer

  for token in marged_tokens:
    if token not in vocab:
      vocab[token] = len(vocab)

In [34]:
df.apply(build_vocab,axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
166,None
167,None
168,None
169,None


In [35]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france?': 6,
 'paris': 7,
 'germany?': 8,
 'berlin': 9,
 'italy?': 10,
 'rome': 11,
 'spain?': 12,
 'madrid': 13,
 'japan?': 14,
 'tokyo': 15,
 'canada?': 16,
 'ottawa': 17,
 'brazil?': 18,
 'brasilia': 19,
 'australia?': 20,
 'canberra': 21,
 'india?': 22,
 'new': 23,
 'delhi': 24,
 'china?': 25,
 'beijing': 26,
 'russia?': 27,
 'moscow': 28,
 'united': 29,
 'states?': 30,
 'washington': 31,
 'dc': 32,
 'mexico?': 33,
 'mexico': 34,
 'city': 35,
 'egypt?': 36,
 'cairo': 37,
 'turkey?': 38,
 'ankara': 39,
 'argentina?': 40,
 'buenos': 41,
 'aires': 42,
 'south': 43,
 'korea?': 44,
 'seoul': 45,
 'indonesia?': 46,
 'jakarta': 47,
 'pakistan?': 48,
 'islamabad': 49,
 'bangladesh?': 50,
 'dhaka': 51,
 'nepal?': 52,
 'kathmandu': 53,
 'sri': 54,
 'lanka?': 55,
 'colombo': 56,
 'thailand?': 57,
 'bangkok': 58,
 'malaysia?': 59,
 'kuala': 60,
 'lumpur': 61,
 'vietnam?': 62,
 'hanoi': 63,
 'uae?': 64,
 'abu': 65,
 'dhabi

In [45]:
def text_to_indices(text,vocab):
  indexed_text = []



  for token in tokenize(text):
    # print(token)
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])

  return indexed_text

In [46]:
text_to_indices("What is Phitron",vocab)

[1, 2, 0]

# Dataset and Dataloader

In [48]:
import torch
from torch.utils.data import Dataset,DataLoader

In [49]:
index = 0
text_to_indices(df.iloc[index]['question'],vocab)

[1, 2, 3, 4, 5, 6]

In [51]:
class QADataset(Dataset):
  def __init__(self,df,vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self,index):
    numerical_question = text_to_indices(self.df.iloc[index]['question'],self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'],self.vocab)

    return torch.tensor(numerical_question),torch.tensor(numerical_answer)


In [52]:
dataset = QADataset(df,vocab)


In [54]:
dataset[1]

(tensor([1, 2, 3, 4, 5, 8]), tensor([9]))

In [55]:
dataloader = DataLoader(dataset,batch_size=1,shuffle=True)

In [56]:
for question,answer in dataloader:
  print(question,answer[0])

tensor([[ 94, 101,   2, 102, 103,   3, 104, 105]]) tensor([106])
tensor([[ 96, 241, 243,  94, 216,   2, 217, 218, 219, 220]]) tensor([221])
tensor([[244, 245, 246,   1,   2, 144, 145]]) tensor([146, 147])
tensor([[237, 238,  94, 209, 210, 211]]) tensor([212])
tensor([[ 94, 203,   2, 207]]) tensor([208])
tensor([[244, 245, 246,  80,  85,   3,  86]]) tensor([87, 88, 89])
tensor([[ 1,  2,  3,  4,  5, 48]]) tensor([49])
tensor([[ 96, 241, 243,   1,   2, 127, 123, 124]]) tensor([128, 129])
tensor([[ 96, 241, 243,   1,   2,  90,  91,  92]]) tensor([93])
tensor([[ 94, 187,   2, 188,   3, 189,   5, 190]]) tensor([191])
tensor([[237, 238,  80,  81, 171]]) tensor([ 87, 172])
tensor([[ 1,  2,  3,  4,  5, 71, 72]]) tensor([73])
tensor([[237, 238,  80,  85,   3,  86]]) tensor([87, 88, 89])
tensor([[252, 241, 253, 254,   1,   2,   3, 227, 230, 231,  71,  72]]) tensor([232])
tensor([[237, 238,  94, 216,   2, 217, 218, 219, 220]]) tensor([221])
tensor([[ 1,  2,  3,  4,  5, 57]]) tensor([58])
tensor([[

In [58]:
# squeeze

import torch

x = torch.tensor([[[1,2,3]]])

print(x.shape)

y = x.squeeze(0)
print(y.shape)

torch.Size([1, 1, 3])
torch.Size([1, 3])


# RNN Architecture Implementation

In [66]:
import torch.nn as nn
class simpleRNN(nn.Module):
  def __init__(self,vocab_size,embedding_dim=50,hidden_size=64):
    super().__init__()

    self.embedding = nn.Embedding(vocab_size,embedding_dim)
    self.rnn = nn.RNN(embedding_dim,hidden_size,batch_first=True)
    self.fc = nn.Linear(hidden_size,vocab_size)


  def forward(self,question):
    embedded = self.embedding(question)
    _,final = self.rnn(embedded)

    return self.fc(final.squeeze(0))

In [67]:
model = simpleRNN(vocab_size = len(vocab))

#training loop

In [68]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.0001)
epochs = 50

In [69]:
for epoch in range(epochs):
    total_loss = 0
    for question, answer in dataloader:
        optimizer.zero_grad()
        output = model(question)
        # Fix: take only the first answer token as target → shape (1,)
        target = answer[0][0].unsqueeze(0)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{epochs}  |  Loss: {total_loss:.4f}")

Epoch  10/50  |  Loss: 611.0905
Epoch  20/50  |  Loss: 363.1412
Epoch  30/50  |  Loss: 241.4065
Epoch  40/50  |  Loss: 171.0126
Epoch  50/50  |  Loss: 122.2275


#Prediction

In [70]:
vocab.items()

dict_items([('<UNK>', 0), ('what', 1), ('is', 2), ('the', 3), ('capital', 4), ('of', 5), ('france?', 6), ('paris', 7), ('germany?', 8), ('berlin', 9), ('italy?', 10), ('rome', 11), ('spain?', 12), ('madrid', 13), ('japan?', 14), ('tokyo', 15), ('canada?', 16), ('ottawa', 17), ('brazil?', 18), ('brasilia', 19), ('australia?', 20), ('canberra', 21), ('india?', 22), ('new', 23), ('delhi', 24), ('china?', 25), ('beijing', 26), ('russia?', 27), ('moscow', 28), ('united', 29), ('states?', 30), ('washington', 31), ('dc', 32), ('mexico?', 33), ('mexico', 34), ('city', 35), ('egypt?', 36), ('cairo', 37), ('turkey?', 38), ('ankara', 39), ('argentina?', 40), ('buenos', 41), ('aires', 42), ('south', 43), ('korea?', 44), ('seoul', 45), ('indonesia?', 46), ('jakarta', 47), ('pakistan?', 48), ('islamabad', 49), ('bangladesh?', 50), ('dhaka', 51), ('nepal?', 52), ('kathmandu', 53), ('sri', 54), ('lanka?', 55), ('colombo', 56), ('thailand?', 57), ('bangkok', 58), ('malaysia?', 59), ('kuala', 60), ('lum

In [71]:
# index to word
idx_to_word = {idx : word for word,idx in vocab.items() }
idx_to_word

{0: '<UNK>',
 1: 'what',
 2: 'is',
 3: 'the',
 4: 'capital',
 5: 'of',
 6: 'france?',
 7: 'paris',
 8: 'germany?',
 9: 'berlin',
 10: 'italy?',
 11: 'rome',
 12: 'spain?',
 13: 'madrid',
 14: 'japan?',
 15: 'tokyo',
 16: 'canada?',
 17: 'ottawa',
 18: 'brazil?',
 19: 'brasilia',
 20: 'australia?',
 21: 'canberra',
 22: 'india?',
 23: 'new',
 24: 'delhi',
 25: 'china?',
 26: 'beijing',
 27: 'russia?',
 28: 'moscow',
 29: 'united',
 30: 'states?',
 31: 'washington',
 32: 'dc',
 33: 'mexico?',
 34: 'mexico',
 35: 'city',
 36: 'egypt?',
 37: 'cairo',
 38: 'turkey?',
 39: 'ankara',
 40: 'argentina?',
 41: 'buenos',
 42: 'aires',
 43: 'south',
 44: 'korea?',
 45: 'seoul',
 46: 'indonesia?',
 47: 'jakarta',
 48: 'pakistan?',
 49: 'islamabad',
 50: 'bangladesh?',
 51: 'dhaka',
 52: 'nepal?',
 53: 'kathmandu',
 54: 'sri',
 55: 'lanka?',
 56: 'colombo',
 57: 'thailand?',
 58: 'bangkok',
 59: 'malaysia?',
 60: 'kuala',
 61: 'lumpur',
 62: 'vietnam?',
 63: 'hanoi',
 64: 'uae?',
 65: 'abu',
 66: 'd

In [72]:

def predict(question_text,model,vocab,idx_to_word):
  model.eval()

  with torch.no_grad():
    indices = text_to_indices(question_text,vocab)

    if not indices:
      return ""

    x = torch.tensor(indices).unsqueeze(0)  #batch size, sequence legth

    pred = torch.argmax(model(x),dim=1).item()


  return idx_to_word.get(pred,"")


In [73]:

# Test
tests = [
    "What is the capital of France?",
    "What is H2O commonly called?",
    "Which planet is known as the red planet?",
    "Which bird cannot fly?",
    "What is the capital of Japan ?",

]
for q in tests:
    print(f"{q:45} → {predict(q, model, vocab, idx_to_word)}")

What is the capital of France?                → paris
What is H2O commonly called?                  → water
Which planet is known as the red planet?      → mars
Which bird cannot fly?                        → ostrich
What is the capital of Japan ?                → william
